In [ ]:
#@title Prevent disconnections
%%html
<audio src="https://oobabooga.github.io/silence.m4a" controls>

In [ ]:
#@title Setup SwarmUI
import os
SWARMPATH = '/content/'
os.environ['SWARMPATH'] = SWARMPATH
os.environ['SWARM_NO_VENV'] = 'true'

# Prevent dotnet JIT segfaults in the Colab VM (known W^X issue) and trim memory use
os.environ['DOTNET_EnableWriteXorExecute'] = '0'
os.environ['DOTNET_TieredPGO'] = '0'
os.environ['DOTNET_TieredCompilation'] = '0'
os.environ['DOTNET_gcServer'] = '0'
os.environ['MSBUILDDISABLENODEREUSE'] = '1'

!apt install -y aria2

# Colab ships with no swap; add 8G so the dotnet build / SwarmUI don't get OOM-killed
!if ! swapon --show | grep -q swapfile; then fallocate -l 8G /content/swapfile && chmod 600 /content/swapfile && mkswap /content/swapfile && swapon /content/swapfile; fi

# Install dotnet 8.0
!wget -q https://dot.net/v1/dotnet-install.sh -O dotnet-install.sh
!chmod +x dotnet-install.sh
!./dotnet-install.sh --channel 8.0

# Install cloudflared for sharing
!wget -q https://github.com/cloudflare/cloudflared/releases/download/2024.8.2/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

%cd $SWARMPATH

# Clone SwarmUI
!git clone https://github.com/mcmonkeyprojects/SwarmUI.git

# Create model directory
!mkdir -p /content/SwarmUI/Models/checkpoints

In [ ]:
#@title Select & Download Model
KEY = ""  # @param {type:"string"}
KEY = "token=" + KEY.strip()

MODELS = {
    "deepDarkHentaiMixNSFW_v61Hybrid": {
        "file": "deepDarkHentaiMixNSFW_v61Hybrid.safetensors",
        "url": f"https://civitai.com/api/download/models/634653?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
    "cyberrealisticPony_semiRealV40": {
        "file": "cyberrealisticPony_semiRealV40.safetensors",
        "url": f"https://civitai.com/api/download/models/2268768?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
    "novaCartoon_v10": {
        "file": "novaCartoon_v10.safetensors",
        "url": f"https://civitai.com/api/download/models/821389?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
}

CHOICE = "cyberrealisticPony_semiRealV40"  # @param ["cyberrealisticPony_semiRealV40","deepDarkHentaiMixNSFW_v61Hybrid","novaCartoon_v10"]

MODEL = MODELS[CHOICE]
MODEL_PATH = "/content/SwarmUI/Models/checkpoints"
USER_AGENT = '"User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64)"'

!aria2c --enable-http-keep-alive=false --header=$USER_AGENT --console-log-level=error -c -x 16 -s 16 -k 1M --summary-interval=5 -d $MODEL_PATH -o {MODEL['file']} "{MODEL['url']}"

In [ ]:
#@title Launch SwarmUI
import os
# Re-apply dotnet segfault workarounds in case the kernel was restarted since setup
os.environ['DOTNET_EnableWriteXorExecute'] = '0'
os.environ['DOTNET_TieredPGO'] = '0'
os.environ['DOTNET_TieredCompilation'] = '0'
os.environ['DOTNET_gcServer'] = '0'
os.environ['MSBUILDDISABLENODEREUSE'] = '1'

%cd /content/SwarmUI

!git fetch
!git reset --hard origin/master
!git pull --autostash
!rm -rf ./src/bin/live_release

# Build explicitly with a single MSBuild node (low memory, avoids the Colab segfault);
# launch-linux.sh will then skip the build since the DLL is already present.
!dotnet build src/SwarmUI.csproj --configuration Release -o ./src/bin/live_release -m:1

!bash ./launch-linux.sh --launch_mode none --cloudflared-path cloudflared